# Day K3 — EMG와 피부전도도를 NeuroKit2로: 라이브러리 기본값 읽기와 단위의 한계

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/K-multimodal-fog/day-K3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/K-multimodal-fog/day-K3.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q neurokit2

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 손 구현 vs `nk.emg_process`: 모양은 닮았는데 비교의 방향이 다르다

In [ ]:
import numpy as np
import neurokit2 as nk
from scipy.signal import butter, sosfiltfilt

# ---- 합성 데이터 뼈대 + 전경골근 EMG (Day K1 예시 2와 같은 가정) ----
fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)
burst = (np.sin(2*np.pi*1.0*t) > 0.5).astype(float)
amp = np.where(label == 0, 0.1 + 0.9*burst, 0.5)
emg = amp * rng.standard_normal(t.size)

# (1) Day I5 방식 손 구현: band-pass → 전파 정류 → low-pass envelope
def hand_envelope(x, fs, band=(20, 240), env_fc=6):
    bp = sosfiltfilt(butter(4, band, btype="bandpass", fs=fs, output="sos"), x - x.mean())
    return sosfiltfilt(butter(2, env_fc, btype="low", fs=fs, output="sos"), np.abs(bp))
env_hand = hand_envelope(emg, fs)

# (2) NeuroKit2 기본값
signals, info = nk.emg_process(emg, sampling_rate=fs)
env_nk = signals["EMG_Amplitude"].to_numpy()
print("nk.emg_process 열:", list(signals.columns))
print("두 envelope의 상관 r = %.3f" % np.corrcoef(env_hand, env_nk)[0, 1])
print("평균 크기: 손 %.3f, nk %.3f" % (env_hand.mean(), env_nk.mean()))
for name, e in [("손", env_hand), ("nk", env_nk)]:
    print(f"{name}: FOG 구간 평균 / 걷기 구간 평균 = {e[label == 1].mean() / e[label == 0].mean():.2f}")

# (3) 같은 설정이면 같은 숫자인가: nk.signal_filter로 손 구현의 band-pass를 다시 만든다
bp_scipy = sosfiltfilt(butter(4, (20, 240), btype="bandpass", fs=fs, output="sos"), emg - emg.mean())
bp_nk = nk.signal_filter(emg - emg.mean(), sampling_rate=fs, lowcut=20, highcut=240, method="butterworth", order=4)
print("같은 설정의 band-pass 최대 차이: %.2e" % np.max(np.abs(bp_scipy - bp_nk)))

### 예시 2 — window별 RMS: RMS와 MAV가 다른 답을 낼 때

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)
burst = (np.sin(2*np.pi*1.0*t) > 0.5).astype(float)
amp = np.where(label == 0, 0.1 + 0.9*burst, 0.5)
emg = amp * rng.standard_normal(t.size)

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):      # Day K2에서 만든 함수
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

bp = sosfiltfilt(butter(4, (20, 240), btype="bandpass", fs=fs, output="sos"), emg - emg.mean())
W = 2 * fs
st = window_starts(bp.size, fs)
seg = bp[st[:, None] + np.arange(W)]                        # (window 수, 1000)
lab = label[st[:, None] + np.arange(W)]
win = pd.DataFrame({"start_s": st / fs,
                    "rms": np.sqrt(np.mean(seg ** 2, axis=1)),          # 제곱 → 평균 → 제곱근
                    "mav": np.mean(np.abs(seg), axis=1),                # 비교용: 절댓값 평균
                    "cv_env": [np.std(e) / np.mean(e) for e in np.abs(seg)],  # 리듬이 있으면 크다
                    "fog": (lab.mean(axis=1) >= 0.5).astype(int)})
print(win.groupby("fog")[["rms", "mav", "cv_env"]].median().round(3))
print("RMS / MAV 비 (가우스 잡음이면 약 1.25):", round(float((win.rms / win.mav).median()), 3))

### 예시 3 — SC 분해, 10 Hz로 내리기, 그리고 부호를 모를 때

In [ ]:
import numpy as np
import neurokit2 as nk
from scipy.signal import resample_poly

fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)
# ---- 오늘 덧붙임: 피부전도도 SC (µS라고 가정한 설명용 신호) ----
tonic = 5 + 0.01 * t                         # SCL: 기록 내내 천천히 오르는 기저선(가정)
scr_onsets = [8.0, 23.0, 44.0]               # SCR: FOG 시작 2초 전(23초)에 하나 있다고 가정
phasic = np.zeros(t.size)
for s in scr_onsets:
    tau = np.clip(t - s, 0, None)
    phasic += 0.4 * (np.exp(-tau / 2.0) - np.exp(-tau / 0.5)) * (t >= s)   # 빠른 상승·느린 하강
sc = tonic + phasic + 0.005 * rng.standard_normal(t.size)

def summarize(x, fs, name):
    sig, info = nk.eda_process(x, sampling_rate=fs)
    peaks = np.flatnonzero(sig["SCR_Peaks"].to_numpy()) / fs
    print(f"{name:16s} 행 {x.size:5d} | SCR 봉우리(초): {np.round(peaks, 1).tolist()} "
          f"| tonic 처음·끝: {sig['EDA_Tonic'].iloc[0]:.2f} → {sig['EDA_Tonic'].iloc[-1]:.2f}")
    return sig

sig = summarize(sc, fs, "500 Hz")
print("열:", list(sig.columns))
sc10 = resample_poly(sc, up=1, down=50, padtype="line")    # 저역 통과 후 500 → 10 Hz
summarize(sc10, 10, "10 Hz (필터 후 내림)")
summarize(2000 - 100 * sc, fs, "부호 뒤집힌 카운트")        # 회로에 따라 카운트가 거꾸로 움직일 수 있다(가정)

## 직접 해보기

모든 문제는 위 합성 데이터 뼈대(`fs`, `t`, `label`, `rng`, `acc`)와 예시 1의 `emg`를 쓴다고 가정합니다. 문제 앞에 다음을 먼저 실행하세요.

In [ ]:
import numpy as np
import neurokit2 as nk
from scipy.signal import butter, sosfiltfilt, resample_poly
fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)
burst = (np.sin(2*np.pi*1.0*t) > 0.5).astype(float)
amp = np.where(label == 0, 0.1 + 0.9*burst, 0.5)
emg = amp * rng.standard_normal(t.size)

**1. RMS 한 줄.** 처음 2초(1000표본)의 RMS를 구하세요.

In [ ]:
seg = emg[:1000].reshape(1, -1)
print("첫 2초 RMS: %.3f" % np.sqrt(np.mean(seg ** ____, axis=1))[0])

In [ ]:
print("첫 2초 RMS: %.3f" % np.sqrt(np.mean(seg ** 2, axis=1))[0])

**2. 전파 정류.** band-pass한 EMG의 평균은 거의 0입니다. 정류해서 평균을 다시 구하세요.

In [ ]:
bp = sosfiltfilt(butter(4, (20, 240), btype="bandpass", fs=fs, output="sos"), emg - emg.mean())
print("정류 전 평균 %.4f, 정류 후 평균 %.3f" % (bp.mean(), np.____(bp).mean()))

In [ ]:
print("정류 전 평균 %.4f, 정류 후 평균 %.3f" % (bp.mean(), np.abs(bp).mean()))

**3. NeuroKit2로 처리하고 FOG/걷기 비 구하기.**

In [ ]:
signals, info = nk.emg_process(emg, sampling_rate=____)
a = signals["EMG_Amplitude"].to_numpy()
print("nk amplitude FOG/걷기 = %.2f" % (a[label == 1].mean() / a[label == 0].mean()))

In [ ]:
signals, info = nk.emg_process(emg, sampling_rate=fs)

**4. SC를 10 Hz로 내리기.** 500 Hz SC를 10 Hz로 내리려면 몇 분의 1로 줄입니까?

In [ ]:
sc = 5 + 0.01 * t + 0.005 * rng.standard_normal(t.size)
sc10 = resample_poly(sc, up=1, down=____, padtype="line")
print(sc.size, "→", sc10.size, "행")

In [ ]:
sc10 = resample_poly(sc, up=1, down=50, padtype="line")

**5. 과제 안 z-점수.** 단위가 카운트인 tonic 값을 과제 안에서 표준화하세요.

In [ ]:
tonic = np.array([1500.0, 1510.0, 1520.0, 1530.0, 1540.0])
z = (tonic - tonic.mean()) / tonic.____()
print(np.round(z, 2))

In [ ]:
z = (tonic - tonic.mean()) / tonic.std()

### 실제 데이터로 해보기 (선택)

**R1 — 001의 왼쪽 전경골근: 스펙트럼, 손 구현 vs NeuroKit2, 움직임 잡음 점검.**

In [ ]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.signal import butter, sosfiltfilt, welch

FIL = "data/xuanwu/Filtered Data/001"
fs = 500
d = pd.read_csv(f"{FIL}/task_1.txt", header=None, usecols=[28, 32, 33, 34, 60])   # 28 = 001의 L-TA
emg, y = d[28].to_numpy(), d[60].to_numpy()

# 1) 스펙트럼: EMG의 power가 어느 대역에 있나
f, p = welch(emg - emg.mean(), fs=fs, nperseg=1000)
tot = p[f > 0].sum()
for lo, hi in [(0, 10), (10, 20), (20, 50), (50, 250)]:
    print(f"{lo:3d}–{hi:<3d} Hz: {100 * p[(f > lo) & (f <= hi)].sum() / tot:5.1f}%")

# 2) 손 구현(Day I5 방식) vs nk.emg_process
def hand_envelope(x, fs, band=(20, 240), env_fc=6):
    bp = sosfiltfilt(butter(4, band, btype="bandpass", fs=fs, output="sos"), x - x.mean())
    return sosfiltfilt(butter(2, env_fc, btype="low", fs=fs, output="sos"), np.abs(bp))
env_hand = hand_envelope(emg, fs)
env_nk = nk.emg_process(emg, sampling_rate=fs)[0]["EMG_Amplitude"].to_numpy()
print("손 vs nk envelope 상관 r = %.3f" % np.corrcoef(env_hand, env_nk)[0, 1])

# 3) 움직임 잡음 의심: EMG envelope이 정강이 충격 덩어리와 얼마나 닮았나
sos_hp = butter(2, 0.5, btype="high", fs=fs, output="sos")
sos_lp = butter(2, 5, btype="low", fs=fs, output="sos")
impact = sosfiltfilt(sos_lp, np.abs(sosfiltfilt(sos_hp, np.linalg.norm(d[[32, 33, 34]].to_numpy(), axis=1))))
for name, e in [("손", env_hand), ("nk", env_nk)]:
    print(f"{name} envelope vs 정강이 충격 r = {np.corrcoef(e, impact)[0, 1]:.3f}")
for name, e in [("손", env_hand), ("nk", env_nk)]:
    print(f"{name}: FOG 구간 평균 / 비FOG 구간 평균 = {e[y == 1].mean() / e[y == 0].mean():.2f}")

**R2 — 기록별 전경골근 RMS: FOG window vs 비FOG window.** 2초 window, majority 규칙(K2)으로 window 라벨을 정하고, 파일 값 그대로의 RMS와 20–240 Hz band-pass 후의 RMS를 비교합니다.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

DATA = "data/xuanwu/Filtered Data"
fs, W = 500, 1000
L_TA = {"001": 28, "002": 28, "003": 27, "004": 27, "005": 27, "006": 28, "007": 28,
        "008/OFF_1": 28, "008/OFF_2": 31, "009": 27, "010": 27, "011": 27, "012": 27}
sos_bp = butter(4, (20, 240), btype="bandpass", fs=fs, output="sos")

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

rows = []
for path in sorted(glob.glob(f"{DATA}/**/task_*.txt", recursive=True)):
    rec = os.path.dirname(os.path.relpath(path, DATA))
    d = pd.read_csv(path, header=None, usecols=[L_TA[rec], 60])
    x = d[L_TA[rec]].to_numpy(); y = d[60].to_numpy()
    xb = sosfiltfilt(sos_bp, x - x.mean())
    st = window_starts(len(y), fs)
    idx = st[:, None] + np.arange(W)
    rows.append(pd.DataFrame({"rec": rec, "fog": (y[idx].mean(1) >= 0.5).astype(int),
                              "rms_file": np.sqrt(np.mean((x[idx] - x[idx].mean(1, keepdims=True)) ** 2, 1)),
                              "rms_bp": np.sqrt(np.mean(xb[idx] ** 2, 1))}))
win = pd.concat(rows, ignore_index=True)
tab = win.groupby(["rec", "fog"])[["rms_file", "rms_bp"]].median().unstack("fog")
out = pd.DataFrame({"FOG windows": win.groupby("rec")["fog"].sum(),
                    "ratio_file": tab[("rms_file", 1)] / tab[("rms_file", 0)],
                    "ratio_bp": tab[("rms_bp", 1)] / tab[("rms_bp", 0)]})
print("L-TA RMS 중앙값 비 (FOG window / 비FOG window), 2초 window, majority 규칙")
print(out.round(2).to_string())
print("비 < 1인 기록 수: 파일 그대로 %d / 20–240 Hz 후 %d (FOG 있는 기록 %d개)" % (
    (out.ratio_file < 1).sum(), (out.ratio_bp < 1).sum(), out["ratio_bp"].notna().sum()))

**R3 — 피부전도도: 과제 안 tonic의 변화와 SCR 빈도.** 팔 센서가 있는 기록만, 10 Hz로 내려 `nk.eda_process`로 처리합니다.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.signal import resample_poly

DATA = "data/xuanwu/Filtered Data"
SC_COL, LABEL = 59, 60                      # Arm의 7번째 열 = SC (ADC 카운트)
fs_sc = 10                                  # SC는 느린 신호라 10 Hz면 충분

rows = []
for path in sorted(glob.glob(f"{DATA}/**/task_*.txt", recursive=True)):
    rec = os.path.dirname(os.path.relpath(path, DATA))
    d = pd.read_csv(path, header=None, usecols=[SC_COL, LABEL])
    sc = d[SC_COL].to_numpy()
    if np.all(sc == sc[0]):                 # 팔 센서가 없으면 SC가 상수(0)
        continue
    sc10 = resample_poly(sc, up=1, down=50, padtype="line")   # 500 → 10 Hz
    y10 = d[LABEL].to_numpy()[::50][: sc10.size]
    sig, info = nk.eda_process(sc10, sampling_rate=fs_sc)
    tonic = sig["EDA_Tonic"].to_numpy()
    z = (tonic - tonic.mean()) / tonic.std()                  # 과제 안에서 z-점수(단위가 카운트라서)
    pos = np.linspace(0, 1, sc10.size)                        # 과제 안의 상대 위치(0 시작, 1 끝)
    peaks = sig["SCR_Peaks"].to_numpy() == 1
    rows.append(pd.DataFrame({"rec": rec, "fog": y10, "z_tonic": z, "pos": pos, "peak": peaks,
                              "sc_raw": sc10}))
s = pd.concat(rows, ignore_index=True)
g = s.groupby(["rec", "fog"])
tab = pd.DataFrame({
    "FOG분": s[s.fog == 1].groupby("rec").size() / fs_sc / 60,
    "z_tonic FOG−비FOG": g["z_tonic"].mean().unstack()[1] - g["z_tonic"].mean().unstack()[0],
    "상대위치 FOG": g["pos"].mean().unstack()[1],
    "SCR/분 FOG": g["peak"].sum().unstack()[1] / (g.size().unstack()[1] / fs_sc / 60),
    "SCR/분 비FOG": g["peak"].sum().unstack()[0] / (g.size().unstack()[0] / fs_sc / 60)})
print(tab.round(2).to_string())
print("\n상관(모든 기록 합쳐서): z_tonic ~ 과제 안 상대위치 r = %.2f" % np.corrcoef(s.z_tonic, s.pos)[0, 1])
print("SC 기록 수:", s.rec.nunique(), "| 기록별 SC 범위(카운트):",
      {r: (int(x.sc_raw.min()), int(x.sc_raw.max())) for r, x in s.groupby("rec")})